# Kaggle Inference Notebook

Run full competition inference on Kaggle-hosted `test` data.

This notebook follows the project requirement that `test` should be evaluated through Kaggle rather than local partial downloads.

In [ ]:
from pathlib import Path

REPO_ZIP_URL = "https://github.com/<owner>/<repo>/archive/refs/heads/main.zip"
WEIGHTS_URL = "https://github.com/<owner>/<repo>/releases/download/<tag>/best.pt"
CONFIG_PATH_IN_REPO = "configs/baseline_quick.yaml"
BATCH_SIZE = 32
OUTPUT_CSV = Path("/kaggle/working/submission.csv")

if "<owner>" in REPO_ZIP_URL or "<owner>" in WEIGHTS_URL:
    raise ValueError("Set REPO_ZIP_URL and WEIGHTS_URL before running the notebook.")

In [ ]:
import shutil
import sys
import urllib.request
import zipfile

import pandas as pd
import torch
import yaml
from torch.utils.data import DataLoader
from functools import partial


WORKDIR = Path("/kaggle/working/asr_numbers_repo")
ZIP_PATH = Path("/kaggle/working/repo.zip")
WEIGHTS_PATH = Path("/kaggle/working/best.pt")
INPUT_ROOT = Path("/kaggle/input")


def download_file(url: str, target: Path) -> Path:
    target.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(url, target)
    return target


def unpack_repo(zip_path: Path, output_dir: Path) -> Path:
    if output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(output_dir)
    children = [path for path in output_dir.iterdir() if path.is_dir()]
    if len(children) != 1:
        raise RuntimeError(f"Expected one extracted repo directory, got {children}")
    return children[0]


def locate_test_split(input_root: Path) -> tuple[Path, Path]:
    candidates = sorted(input_root.rglob("test.csv"))
    for csv_path in candidates:
        root = csv_path.parent
        frame = pd.read_csv(csv_path)
        if "filename" not in frame.columns or len(frame) == 0:
            continue
        first_path = root / frame.iloc[0]["filename"]
        if first_path.exists():
            return csv_path, root
    raise FileNotFoundError(f"Could not locate a valid Kaggle competition test split under {input_root}")


download_file(REPO_ZIP_URL, ZIP_PATH)
repo_root = unpack_repo(ZIP_PATH, WORKDIR)
download_file(WEIGHTS_URL, WEIGHTS_PATH)

sys.path.insert(0, str(repo_root / "src"))

from asr_numbers.dataset import NumbersDataset, collate_batch
from asr_numbers.decoder import decode_number_predictions
from asr_numbers.model import ConvGRUCTCModel
from asr_numbers.vocab import WordVocabulary

test_csv, audio_root = locate_test_split(INPUT_ROOT)
print(f"repo_root={repo_root}")
print(f"weights={WEIGHTS_PATH}")
print(f"test_csv={test_csv}")
print(f"audio_root={audio_root}")

In [ ]:
config = yaml.safe_load((repo_root / CONFIG_PATH_IN_REPO).read_text())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vocab = WordVocabulary()

checkpoint = torch.load(WEIGHTS_PATH, map_location=device)
model = ConvGRUCTCModel(vocab_size=vocab.size, **config["model"]).to(device)
model.load_state_dict(checkpoint["model_state"])
model.eval()

dataset = NumbersDataset(
    csv_path=test_csv,
    audio_root=audio_root,
    sample_rate=int(config["model"]["sample_rate"]),
    with_labels=False,
)
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    collate_fn=partial(collate_batch, vocab=vocab),
)

rows = []
with torch.no_grad():
    for batch in loader:
        waveforms = batch["waveforms"].to(device)
        waveform_lengths = batch["waveform_lengths"].to(device)
        logits, output_lengths = model(waveforms, waveform_lengths)
        predictions = decode_number_predictions(logits.cpu(), output_lengths.cpu(), vocab)
        for filename, transcription in zip(batch["filenames"], predictions, strict=True):
            rows.append({"filename": filename, "transcription": transcription})

submission = pd.DataFrame(rows, columns=["filename", "transcription"])
submission.to_csv(OUTPUT_CSV, index=False)
submission.head()

In [ ]:
print(f"Wrote {len(submission)} rows to {OUTPUT_CSV}")
print(submission.sample(min(5, len(submission)), random_state=42).to_string(index=False))